In [1]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:256"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!pip install -q huggingface_hub datasets

In [2]:
import gc, json, time, threading, subprocess, psutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from huggingface_hub import hf_hub_download
from datasets import load_dataset
from tqdm.auto import tqdm


try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
    print("[INFO] Google Drive mounted.")
except ImportError:
    IN_COLAB = False
    print("[WARN] Not in Colab.")

_DRIVE_ROOT = "/content/drive/MyDrive/tarecmind"

VARIANT_NAME = "lightgcn_warm_only"

CFG = {
    "REPO_ID":       "chuongdo1104/amazon-2023-gold",
    "SILVER_REPO_ID":"chuongdo1104/amazon-2023-silver",
    "DRIVE_ROOT":    _DRIVE_ROOT,
    "DATA_DIR":      f"{_DRIVE_ROOT}/data_{VARIANT_NAME}",
    "SAVE_DIR":      f"{_DRIVE_ROOT}/weights_{VARIANT_NAME}",
    "CACHE_DIR":     "/content/recsys_cache",
    "EMBED_DIM":     128,
    "LLM_DIM":       384,
    "GCN_LAYERS":    2,
    "TEMPERATURE":   0.2,
    "EPOCHS":        50,
    "LR_JOINT":      1e-3,
    "WEIGHT_DECAY":  1e-4,
    "GRAD_CLIP":     1.0,
    "LAMBDA_ALIGN":  0.3,
    "LAMBDA_CL":     0.005,
    "NOISE_SCALE":   0.08,
    "PATIENCE":      10,
    "EVAL_EVERY":    2,
    "ENCODE_CHUNK":  30000,
    "ENCODE_BATCH":  256,
    "SAVE_EVERY_EPOCH": True,
    "KEEPALIVE_MINS":   25,
    "WARM_ONLY":     True,
    "EVAL_STRAT_GROUPS": {
        "HEAD": 2500, "MID": 2500, "TAIL": 2500,
    },
}

for d in [CFG["DATA_DIR"], CFG["SAVE_DIR"], CFG["CACHE_DIR"]]:
    os.makedirs(d, exist_ok=True)

print(f"[INFO] Variant: {VARIANT_NAME}")

Mounted at /content/drive
[INFO] Google Drive mounted.
[INFO] Variant: lightgcn_warm_only


In [3]:
def detect_and_adjust_gpu():
    sys_ram_gb = psutil.virtual_memory().total / 1e9
    print(f"[INFO] System RAM: {sys_ram_gb:.1f} GB")
    if not torch.cuda.is_available():
        print("[WARN] No GPU.")
        return "cpu"
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[INFO] GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")
    if vram_gb > 70:
        CFG["BATCH_SIZE"]     = 32768
        CFG["ALIGN_SUBBATCH"] = 8192
        CFG["ACCUM_STEPS"]    = 1
        CFG["ENCODE_BATCH"]   = 1024
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
    elif vram_gb > 35:
        CFG["BATCH_SIZE"]     = 16384
        CFG["ALIGN_SUBBATCH"] = 4096
        CFG["ACCUM_STEPS"]    = 2
        CFG["ENCODE_BATCH"]   = 512
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
    elif vram_gb > 16:
        CFG["BATCH_SIZE"]     = 4096
        CFG["ALIGN_SUBBATCH"] = 1024
        CFG["ACCUM_STEPS"]    = 4
        CFG["ENCODE_BATCH"]   = 512
    else:
        CFG["BATCH_SIZE"]     = 2048
        CFG["ALIGN_SUBBATCH"] = 512
        CFG["ACCUM_STEPS"]    = 4
        CFG["ENCODE_BATCH"]   = 256
    print(f"   -> batch={CFG['BATCH_SIZE']}, accum={CFG.get('ACCUM_STEPS',1)}")
    return "cuda"

device = detect_and_adjust_gpu()

_keepalive_stop = threading.Event()
def _keepalive_worker():
    interval = CFG["KEEPALIVE_MINS"] * 60
    ka_file  = os.path.join(CFG["CACHE_DIR"], "keepalive.txt")
    while not _keepalive_stop.is_set():
        time.sleep(interval)
        if not _keepalive_stop.is_set():
            with open(ka_file, "w") as f:
                f.write(f"alive {time.strftime('%H:%M:%S')}\n")
threading.Thread(target=_keepalive_worker, daemon=True).start()
print(f"[INFO] Keep-alive started. Device: {device}")

[INFO] System RAM: 89.6 GB
[INFO] GPU: NVIDIA A100-SXM4-40GB | VRAM: 42.4 GB
   -> batch=16384, accum=2
[INFO] Keep-alive started. Device: cuda


In [4]:
POP_GROUP  = {0: "HEAD", 1: "MID", 2: "TAIL"}
USER_GROUP = {0: "INACTIVE", 1: "ACTIVE", 2: "SUPER_ACTIVE"}

def download_hf(filename, local_dir=None):
    target     = local_dir or CFG["CACHE_DIR"]
    local_path = os.path.join(target, os.path.basename(filename))
    if os.path.exists(local_path):
        return local_path
    print(f"[INFO] Downloading {filename}...")
    return hf_hub_download(
        repo_id=CFG["REPO_ID"], filename=filename,
        repo_type="dataset", local_dir=target,
    )

print("\n--- LOADING GOLD METADATA ---")
with open(download_hf("gold/gold_dataset_stats.json"), "r") as f:
    dataset_stats = json.load(f)

num_users   = dataset_stats["n_users"]
num_items   = dataset_stats["n_items"]
total_nodes = num_users + num_items
print(f"[INFO] {num_users:,} users | {num_items:,} items | {total_nodes:,} total nodes")
print(f"[INFO] Sparsity: {dataset_stats.get('sparsity_pct','N/A')}")

edge_index_raw   = np.load(download_hf("gold/gold_edge_index.npy"))
train_edge_index = torch.from_numpy(edge_index_raw).long()
n_train_edges    = train_edge_index.shape[1]
del edge_index_raw

item_train_freq_np = np.load(download_hf("gold/gold_item_train_freq.npy"))
item_pop_group_np  = np.load(download_hf("gold/gold_item_popularity_group.npy"))
item_train_freq_t  = torch.from_numpy(item_train_freq_np).long()
item_pop_group_t   = torch.from_numpy(item_pop_group_np.astype("int32")).long()

assert n_train_edges > 0,              "Edge list empty!"
assert len(item_train_freq_np) == num_items

# ============================================================
# WARM-ONLY SETUP: bỏ hoàn toàn cold-start khỏi bài toán
# Warm item = item có ít nhất 1 tương tác trong train.
# Cold item = item có train_freq == 0.
# Các bước sau sẽ chỉ dùng warm item cho:
#   - positive validation/test
#   - candidate ranking
#   - negative sampling
#   - Coverage denominator
# ============================================================
item_train_freq_cpu = item_train_freq_t.detach().cpu()
warm_item_mask_cpu  = item_train_freq_cpu > 0
cold_item_mask_cpu  = ~warm_item_mask_cpu
warm_item_ids_cpu   = torch.where(warm_item_mask_cpu)[0].long()
num_warm_items      = int(warm_item_mask_cpu.sum().item())
num_cold_items      = int(cold_item_mask_cpu.sum().item())

assert num_warm_items > 0, "No warm items found!"
assert num_warm_items + num_cold_items == num_items

print(f"\n[INFO] {n_train_edges:,} train edges")
for code, name in POP_GROUP.items():
    cnt = int((item_pop_group_t == code).sum())
    warm_cnt = int(((item_pop_group_t == code).cpu() & warm_item_mask_cpu).sum())
    print(f"  {name:10s}: {cnt:,} total | {warm_cnt:,} warm")
print(f"  {'COLD_REMOVED':10s}: {num_cold_items:,} ({num_cold_items/num_items*100:.1f}% catalog)")
print(f"[INFO] Warm-only catalog: {num_warm_items:,}/{num_items:,} items")

del item_train_freq_np, item_pop_group_np
gc.collect()
print("[OK] Gold data loaded. Warm/cold masks prepared.")


--- LOADING GOLD METADATA ---
[INFO] Downloading gold/gold_dataset_stats.json...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


gold_dataset_stats.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

[INFO] 1,847,662 users | 1,610,012 items | 3,457,674 total nodes
[INFO] Sparsity: 99.9995%
[INFO] Downloading gold/gold_edge_index.npy...


gold/gold_edge_index.npy:   0%|          | 0.00/223M [00:00<?, ?B/s]

[INFO] Downloading gold/gold_item_train_freq.npy...


gold/gold_item_train_freq.npy:   0%|          | 0.00/12.9M [00:00<?, ?B/s]

[INFO] Downloading gold/gold_item_popularity_group.npy...


gold/gold_item_popularity_group.npy:   0%|          | 0.00/1.61M [00:00<?, ?B/s]


[INFO] 13,964,281 train edges
  HEAD      : 210,925 total | 210,925 warm
  MID       : 116,283 total | 116,283 warm
  TAIL      : 714,913 total | 714,913 warm
  COLD_REMOVED: 567,891 (35.3% catalog)
[INFO] Warm-only catalog: 1,042,121/1,610,012 items
[OK] Gold data loaded. Warm/cold masks prepared.


In [5]:
path_train_edges = os.path.join(CFG["DATA_DIR"], "train_edges.pt")
path_val_edges   = os.path.join(CFG["DATA_DIR"], "val_edges.pt")
path_sparse_adj  = os.path.join(CFG["DATA_DIR"], "sparse_adj.pt")

# -- Val edges --
if os.path.exists(path_val_edges):
    print("[INFO] Loading val_edges from Drive cache...")
    val_edges_t = torch.load(path_val_edges, map_location="cpu", weights_only=True)
    VAL_SIZE    = val_edges_t.shape[1]
else:
    print("[INFO] Building val_edges from silver_val_ground_truth.parquet...")
    df_val_gt = load_dataset(
        CFG["SILVER_REPO_ID"],
        data_dir="silver/silver_val_ground_truth.parquet",
        split="train",
    ).to_pandas()
    path_item_map = download_hf("gold/gold_item_id_map.parquet")
    path_user_map = download_hf("gold/gold_user_id_map.parquet")
    df_item_map = pd.read_parquet(path_item_map, columns=["parent_asin", "item_idx"])
    df_user_map = pd.read_parquet(path_user_map, columns=["reviewer_id",  "user_idx"])
    df_val_gt = (
        df_val_gt
        .merge(df_item_map, on="parent_asin", how="inner")
        .merge(df_user_map, on="reviewer_id",  how="inner")
    )
    VAL_SIZE    = len(df_val_gt)
    val_edges_t = torch.tensor(
        np.stack([df_val_gt["user_idx"].values.astype(np.int64),
                  df_val_gt["item_idx"].values.astype(np.int64)], axis=0),
        dtype=torch.long,
    )
    torch.save(val_edges_t, path_val_edges)
    del df_val_gt, df_item_map, df_user_map; gc.collect()
print(f"  val_edges raw: {VAL_SIZE:,}")

# -- Warm-only validation positives --
# Loại mọi validation interaction có positive item cold.
# Như vậy checkpoint không còn bị ảnh hưởng bởi cold-start.
val_items_cpu = val_edges_t[1].detach().cpu()
val_valid_item_mask = (val_items_cpu >= 0) & (val_items_cpu < num_items)
val_warm_mask = torch.zeros_like(val_valid_item_mask, dtype=torch.bool)
val_warm_mask[val_valid_item_mask] = warm_item_mask_cpu[val_items_cpu[val_valid_item_mask]]

VAL_SIZE_RAW = int(val_edges_t.shape[1])
val_edges_t = val_edges_t[:, val_warm_mask]
VAL_SIZE = int(val_edges_t.shape[1])
print(f"[FILTER] Val warm-only: {VAL_SIZE_RAW:,} -> {VAL_SIZE:,}")
print(f"[FILTER] Removed cold/invalid val positives: {VAL_SIZE_RAW - VAL_SIZE:,}")
assert (item_train_freq_cpu[val_edges_t[1].cpu()] > 0).all(), "Val still contains cold positives!"

# -- Train edges --
if os.path.exists(path_train_edges):
    train_edge_index_dev = torch.load(path_train_edges, map_location=device, weights_only=True)
else:
    train_edge_index_dev = train_edge_index.to(device)
    torch.save(train_edge_index_dev.cpu(), path_train_edges)
train_edge_index = train_edge_index_dev
n_train_edges    = train_edge_index.shape[1]

# -- Sparse adj (D^-0.5 A D^-0.5) --
if os.path.exists(path_sparse_adj):
    print("[INFO] Loading sparse_adj from Drive cache...")
    sparse_adj = torch.load(path_sparse_adj, map_location=device, weights_only=True)
else:
    print("[INFO] Building sparse_adj...")
    row = torch.cat([train_edge_index[0], train_edge_index[1] + num_users])
    col = torch.cat([train_edge_index[1] + num_users, train_edge_index[0]])
    gei = torch.stack([row, col]).long()
    deg = torch.bincount(gei[0], minlength=total_nodes).float()
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0
    ew  = (deg_inv_sqrt[gei[0]] * deg_inv_sqrt[gei[1]]).half()
    adj = torch.sparse_coo_tensor(gei, ew, (total_nodes, total_nodes))
    sparse_adj = adj.coalesce().to(device).to_sparse_csr()
    torch.save(sparse_adj.cpu(), path_sparse_adj)
    del adj, ew, deg, deg_inv_sqrt, gei, row, col; gc.collect()

# -- Move tensors and masks to device --
item_train_freq_t = item_train_freq_t.to(device)
item_pop_group_t  = item_pop_group_t.to(device)
warm_item_mask    = warm_item_mask_cpu.to(device)
cold_item_mask    = cold_item_mask_cpu.to(device)
warm_item_ids     = warm_item_ids_cpu.to(device)

print(f"[OK] Graph ready. Train: {n_train_edges:,} | Val warm-only: {VAL_SIZE:,}")
print(f"[OK] Candidate set warm-only: {num_warm_items:,} items | Cold removed: {num_cold_items:,}")
gc.collect(); torch.cuda.empty_cache()

[INFO] Loading val_edges from Drive cache...
  val_edges raw: 1,847,662
[FILTER] Val warm-only: 1,847,662 -> 1,774,977
[FILTER] Removed cold/invalid val positives: 72,685
[INFO] Loading sparse_adj from Drive cache...
[OK] Graph ready. Train: 13,964,281 | Val warm-only: 1,774,977
[OK] Candidate set warm-only: 1,042,121 items | Cold removed: 567,891


/usr/local/lib/python3.12/dist-packages/torch/_utils.py:361: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  result = torch.sparse_compressed_tensor(


In [6]:
class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=128, gcn_layers=2):
        super().__init__()
        self.num_users  = num_users
        self.num_items  = num_items
        self.embed_dim  = embed_dim
        self.gcn_layers = gcn_layers
        self.user_id_emb = nn.Embedding(num_users, embed_dim)
        self.item_id_emb = nn.Embedding(num_items, embed_dim)
        nn.init.normal_(self.user_id_emb.weight, std=0.01)
        nn.init.normal_(self.item_id_emb.weight, std=0.01)

    def forward_gcn(self, adj):
        x = torch.cat([self.user_id_emb.weight, self.item_id_emb.weight], dim=0)
        z_accum = x.clone()
        for _ in range(self.gcn_layers):
            x = torch.sparse.mm(adj, x.to(adj.dtype)).to(torch.float32)
            z_accum += x
        return z_accum / (self.gcn_layers + 1.0)

    def forward(self, adj, batch_users, batch_pos, batch_neg):
        z_G = self.forward_gcn(adj)
        user_G, item_G = z_G.split([self.num_users, self.num_items])
        return user_G[batch_users], item_G[batch_pos], item_G[batch_neg]

print("[OK] LightGCN model class defined.")

[OK] LightGCN model class defined.


In [7]:
def bpr_loss(user_emb, pos_emb, neg_emb):
    pos_scores = (user_emb * pos_emb).sum(dim=1)   # dot product thô
    neg_scores = (user_emb * neg_emb).sum(dim=1)
    return -F.logsigmoid(pos_scores - neg_scores).mean()

print("[OK] BPR loss defined.")

[OK] BPR loss defined.


In [8]:
def sample_negatives_uniform(batch_size, warm_item_ids, device):
    """
    Warm-only negative sampling.
    Chỉ lấy item âm trong warm catalog, không lấy cold item.
    """
    if not isinstance(warm_item_ids, torch.Tensor):
        warm_item_ids = torch.tensor(warm_item_ids, dtype=torch.long, device=device)
    warm_item_ids = warm_item_ids.to(device)
    rand_pos = torch.randint(0, warm_item_ids.numel(), (batch_size,), device=device)
    return warm_item_ids[rand_pos]

_neg = sample_negatives_uniform(256, warm_item_ids, device)
assert _neg.shape == (256,)
assert (_neg >= 0).all() and (_neg < num_items).all()
assert (item_train_freq_t[_neg] > 0).all(), "Negative sampler still returns cold items!"
del _neg
print("[OK] Warm-only negative sampling ready. No cold negatives.")

[OK] Warm-only negative sampling ready. No cold negatives.


In [9]:
def group_by_user_items(users, items, deduplicate=True):
    """
    Gom các interaction (user, item) thành dict:
        user_id -> numpy array các positive item của user đó.

    Đây là bước quan trọng để chuyển evaluation từ interaction-level sang user-level:
    - interaction-level: một user có 10 test item thì bị tính 10 lần
    - user-level: một user có 10 test item thì chỉ tính 1 lần, với tập đáp án gồm 10 item
    """
    users_np = np.asarray(users, dtype=np.int64)
    items_np = np.asarray(items, dtype=np.int64)

    if len(users_np) == 0:
        return {}

    df = pd.DataFrame({"u": users_np, "i": items_np})
    if deduplicate:
        df = df.drop_duplicates(["u", "i"])

    grouped = {}
    for u, g in df.groupby("u", sort=False):
        vals = g["i"].values.astype(np.int64)
        grouped[int(u)] = np.unique(vals) if deduplicate else vals
    return grouped


def _build_user_pos_dict_from_eval_groups(eval_groups, warm_item_mask=None, device="cuda"):
    """
    Nhận eval_groups dạng {"FULL": {"u": ..., "i": ...}} hoặc nhiều group,
    sau đó gom lại thành user -> tập positive items.

    Nếu còn positive item cold thì loại bỏ để giữ đúng warm-only protocol.
    """
    all_u, all_i = [], []

    if warm_item_mask is not None:
        warm_item_mask_cpu_local = warm_item_mask.detach().cpu().bool()
    else:
        warm_item_mask_cpu_local = None

    for gname, gdata in eval_groups.items():
        if gdata is None:
            continue

        u = gdata["u"].detach().cpu().long()
        i = gdata["i"].detach().cpu().long()

        if warm_item_mask_cpu_local is not None:
            valid = (i >= 0) & (i < len(warm_item_mask_cpu_local))
            warm = torch.zeros_like(valid, dtype=torch.bool)
            warm[valid] = warm_item_mask_cpu_local[i[valid]]
            keep = valid & warm

            if not bool(keep.all()):
                removed = int((~keep).sum().item())
                print(f"[WARN] {gname}: removed {removed:,} invalid/cold positives before user-level eval.")

            u = u[keep]
            i = i[keep]

        if len(u) > 0:
            all_u.append(u.numpy())
            all_i.append(i.numpy())

    if len(all_u) == 0:
        return {}

    all_u = np.concatenate(all_u)
    all_i = np.concatenate(all_i)
    return group_by_user_items(all_u, all_i, deduplicate=True)


def _filter_eval_positives_seen_in_mask(user_pos_dict, mask_dict):
    """
    An toàn dữ liệu:
    Nếu một positive item trong validation/test cũng nằm trong mask history của chính user,
    item đó sẽ không thể được recommend vì đã bị mask = -inf.
    Khi đó nên loại nó khỏi tập đáp án để metric không bị sai thấp.

    Validation thường mask train.
    Test thường mask train + validation.
    """
    if mask_dict is None:
        return user_pos_dict

    filtered = {}
    removed = 0

    for u, pos_items in user_pos_dict.items():
        seen = mask_dict.get(int(u))
        if seen is None or len(seen) == 0:
            filtered[int(u)] = pos_items
            continue

        seen_np = seen.detach().cpu().numpy().astype(np.int64)
        keep_items = np.setdiff1d(pos_items.astype(np.int64), seen_np, assume_unique=False)
        removed += int(len(pos_items) - len(keep_items))

        if len(keep_items) > 0:
            filtered[int(u)] = keep_items

    if removed > 0:
        print(f"[WARN] Removed {removed:,} eval positives because they were already in the user's mask history.")

    return filtered


def _empty_user_level_accum(Ks, group_names):
    return {
        g: {
            K: {
                "recall_sum": 0.0,
                "ndcg_sum": 0.0,
                "hits": 0.0,
                "n_users": 0,
                "n_pos": 0,
            }
            for K in Ks
        }
        for g in group_names
    }


def _add_user_level_metric(accum, group_name, K, top_items_np, pos_items_np, discounts):
    """
    Cộng metric của 1 user vào accumulator.

    Recall@K(user) = số positive item nằm trong top-K / tổng positive item của user
    NDCG@K(user)   = DCG@K / IDCG@K, với relevance nhị phân.
    """
    if len(pos_items_np) == 0:
        return

    top_k = top_items_np[:K]
    hit_mask = np.isin(top_k, pos_items_np, assume_unique=False)

    hits = float(hit_mask.sum())
    recall = hits / max(len(pos_items_np), 1)

    dcg = float((hit_mask.astype(np.float64) * discounts[:len(top_k)]).sum())
    ideal_len = min(len(pos_items_np), K)
    idcg = float(discounts[:ideal_len].sum()) if ideal_len > 0 else 0.0
    ndcg = dcg / idcg if idcg > 0 else 0.0

    accum[group_name][K]["recall_sum"] += recall
    accum[group_name][K]["ndcg_sum"] += ndcg
    accum[group_name][K]["hits"] += hits
    accum[group_name][K]["n_users"] += 1
    accum[group_name][K]["n_pos"] += int(len(pos_items_np))


def evaluate_lightgcn(model, adj, item_pop_group_t, eval_groups,
                      num_users, num_items, Ks=(20, 40), device="cuda",
                      user_chunk=64, train_mask_dict=None,
                      warm_item_mask=None):
    """
    Warm-only full-ranking USER-LEVEL validation.

    Khác với bản cũ:
    - Bản cũ: mỗi cặp (user, positive item) là một mẫu đánh giá riêng.
    - Bản này: gom tất cả positive item của cùng một user, rồi đánh giá đúng 1 lần/user.

    Với mỗi user:
    1. Tính điểm user với toàn bộ item.
    2. Loại cold item khỏi candidate set.
    3. Mask các item user đã thấy trong train.
    4. Lấy top-K.
    5. So top-K với toàn bộ positive validation items của user.
    """
    model.eval()
    Ks = sorted(list(Ks))
    max_k_requested = max(Ks)

    if warm_item_mask is None:
        warm_item_mask_dev = torch.ones(num_items, dtype=torch.bool, device=device)
        warm_item_mask_cpu_local = torch.ones(num_items, dtype=torch.bool)
    else:
        warm_item_mask_dev = warm_item_mask.to(device).bool()
        warm_item_mask_cpu_local = warm_item_mask.detach().cpu().bool()

    cold_item_mask_dev = ~warm_item_mask_dev
    num_candidates = int(warm_item_mask_dev.sum().item())
    topk_eval = min(max_k_requested, num_candidates)

    user_pos_dict = _build_user_pos_dict_from_eval_groups(
        eval_groups,
        warm_item_mask=warm_item_mask_cpu_local,
        device=device,
    )
    user_pos_dict = _filter_eval_positives_seen_in_mask(user_pos_dict, train_mask_dict)

    eval_user_ids = np.array(list(user_pos_dict.keys()), dtype=np.int64)
    n_eval_users = int(len(eval_user_ids))

    print(f"[USER-LEVEL VAL] Users={n_eval_users:,} | Positives={sum(len(v) for v in user_pos_dict.values()):,}")

    pop_g_cpu = item_pop_group_t.detach().cpu().long()
    POP_NAMES = {2: "TAIL"}
    group_names = ["OVERALL", "TAIL"]
    accum = _empty_user_level_accum(Ks, group_names)
    recommended_mask = {K: torch.zeros(num_items, dtype=torch.bool, device=device) for K in Ks}

    discounts = 1.0 / np.log2(np.arange(2, topk_eval + 2, dtype=np.float64))

    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            z_G_all = model.forward_gcn(adj)
        user_G_all, item_G_all = z_G_all.split([num_users, num_items])
        item_final = item_G_all

        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            for start in tqdm(range(0, n_eval_users, user_chunk), desc="User-level VAL", ncols=100):
                end = min(start + user_chunk, n_eval_users)
                bu_np = eval_user_ids[start:end]
                bu = torch.tensor(bu_np, dtype=torch.long, device=device)

                u_final = user_G_all[bu]
                scores = torch.mm(u_final, item_final.t())
                mask_value = torch.finfo(scores.dtype).min

                # Candidate set warm-only.
                scores[:, cold_item_mask_dev] = mask_value

                # Mask train items.
                if train_mask_dict is not None:
                    for row, uid in enumerate(bu_np):
                        mi = train_mask_dict.get(int(uid))
                        if mi is not None and len(mi) > 0:
                            scores[row, mi.to(device)] = mask_value

                _, topk_idx = torch.topk(scores, topk_eval, dim=1)
                topk_np = topk_idx.detach().cpu().numpy()

                for K in Ks:
                    use_k = min(K, topk_eval)
                    recommended_mask[K][topk_idx[:, :use_k].reshape(-1)] = True

                for row, uid in enumerate(bu_np):
                    pos_items = user_pos_dict[int(uid)].astype(np.int64)
                    top_items = topk_np[row]

                    for K in Ks:
                        _add_user_level_metric(accum, "OVERALL", K, top_items, pos_items, discounts)

                    pos_groups = pop_g_cpu[torch.tensor(pos_items, dtype=torch.long)].numpy()
                    for code, gn in POP_NAMES.items():
                        group_pos = pos_items[pos_groups == code]
                        if len(group_pos) == 0:
                            continue
                        for K in Ks:
                            _add_user_level_metric(accum, gn, K, top_items, group_pos, discounts)

    results = {}
    for grp in group_names:
        results[grp] = {}
        for K in Ks:
            n_users = accum[grp][K]["n_users"]
            results[grp][f"Recall@{K}"] = accum[grp][K]["recall_sum"] / max(n_users, 1)
            results[grp][f"NDCG@{K}"] = accum[grp][K]["ndcg_sum"] / max(n_users, 1)
            results[grp][f"Hits@{K}"] = accum[grp][K]["hits"]
            results[grp][f"n_users@{K}"] = n_users
            results[grp][f"n_pos@{K}"] = accum[grp][K]["n_pos"]

        # Alias để code cũ không lỗi; mặc định lấy K đầu tiên, thường là 20.
        k0 = Ks[0]
        results[grp]["Recall@K"] = results[grp][f"Recall@{k0}"]
        results[grp]["NDCG@K"] = results[grp][f"NDCG@{k0}"]
        results[grp]["n"] = results[grp][f"n_users@{k0}"]

    tail_mask = (item_pop_group_t.to(device) == 2) & warm_item_mask_dev
    n_tail = int(tail_mask.sum().item())

    cov, tcov, apop = {}, {}, {}
    for K in Ks:
        rm = recommended_mask[K] & warm_item_mask_dev
        nr = int(rm.sum().item())
        cov[K] = nr / num_candidates if num_candidates > 0 else 0
        tcov[K] = (rm & tail_mask).sum().item() / n_tail if n_tail > 0 else 0
        apop[K] = item_train_freq_t[rm].float().mean().item() if nr > 0 else 0

    results["Coverage@K"] = cov[Ks[0]]
    results["TailCoverage@K"] = tcov[Ks[0]]
    results["AvgPopularity@K"] = apop[Ks[0]]
    results["CoverageByK"] = cov
    results["TailCoverageByK"] = tcov
    results["AvgPopularityByK"] = apop
    results["NumWarmCandidates"] = num_candidates
    results["EvalLevel"] = "user-level"
    return results


def print_eval_results(epoch, results, Ks=(20, 40)):
    print(f"\n[VAL WARM-ONLY USER-LEVEL] Epoch {epoch:02d}")
    for grp in ["TAIL", "OVERALL"]:
        if grp in results:
            v = results[grp]
            n_users = v.get(f"n_users@{Ks[0]}", v.get("n", 0))
            n_pos = v.get(f"n_pos@{Ks[0]}", 0)
            parts = [f"  {grp:<10} | users={n_users:>8,} | pos={n_pos:>8,}"]
            for K in Ks:
                parts.append(f"R@{K}={v.get(f'Recall@{K}', 0):.4f} | N@{K}={v.get(f'NDCG@{K}', 0):.4f}")
            print(" | ".join(parts))

    cov = results.get("CoverageByK", {})
    tcov = results.get("TailCoverageByK", {})
    apop = results.get("AvgPopularityByK", {})
    for K in Ks:
        print(f"  Coverage@{K}={cov.get(K, 0):.4f} | TailCov@{K}={tcov.get(K, 0):.4f} | AvgPop@{K}={apop.get(K, 0):.1f}")

    print(f"  Warm candidates={results.get('NumWarmCandidates', 0):,}")
    tail = results.get("TAIL", {})
    print(
        f"  [TAIL] R@20={tail.get('Recall@20', 0):.4f} | N@20={tail.get('NDCG@20', 0):.4f} "
        f"| R@40={tail.get('Recall@40', 0):.4f} | N@40={tail.get('NDCG@40', 0):.4f} "
        f"| TailCov@20={tcov.get(20, 0):.4f} | TailCov@40={tcov.get(40, 0):.4f}"
    )

print("[OK] Warm-only USER-LEVEL LightGCN evaluation ready. Cold candidates are masked. Metrics include @20 and @40.")


[OK] Warm-only USER-LEVEL LightGCN evaluation ready. Cold candidates are masked. Metrics include @20 and @40.


In [10]:
class ColabCheckpointManager:
    def __init__(self, weight_dir, data_dir, K=20):
        self.weight_dir = weight_dir
        self.data_dir   = data_dir
        self.K          = K
        os.makedirs(weight_dir, exist_ok=True)
        os.makedirs(data_dir, exist_ok=True)
        self.best_path  = os.path.join(weight_dir, "best.pth")
        self.last_path  = os.path.join(weight_dir, "last.pth")
        self.hist_path  = os.path.join(data_dir, "training_history.json")
        self.proj_path  = os.path.join(data_dir, "z_llm_projected.pt")
        self.eval_path  = os.path.join(data_dir, "eval_sample_groups.pt")

    def save(self, model, optimizer, epoch, metrics, history_list, cfg, is_best=False):
        def _safe(v):
            return v if isinstance(v, (int,float,str,bool,type(None))) else str(v)
        safe_cfg = {k: _safe(v) for k,v in cfg.items() if not k.startswith("_")}
        payload = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "metrics": metrics,
            "config": safe_cfg,
        }
        torch.save(payload, self.last_path)
        if is_best:
            torch.save(payload, self.best_path)
            print(f"[CKPT] NEW BEST saved (Score={metrics.get('Score',0):.4f})")

        current = []
        if os.path.exists(self.hist_path):
            try:
                with open(self.hist_path, "r") as f: current = json.load(f)
            except Exception: pass
        existing_epochs = {h["epoch"] for h in current}
        for entry in history_list:
            if entry.get("epoch") not in existing_epochs:
                current.append(entry)
        current.sort(key=lambda x: x.get("epoch", 0))
        with open(self.hist_path, "w") as f:
            json.dump(current, f, indent=2, ensure_ascii=False)

    def save_projected_llm(self, z):
        torch.save(z.cpu().half(), self.proj_path)

    def load_projected_llm(self, device):
        if not os.path.exists(self.proj_path): return None
        return torch.load(self.proj_path, map_location=device, weights_only=True).float()

    def save_eval_groups(self, g):
        cpu_g = {}
        for k,v in g.items():
            cpu_g[k] = {kk: vv.cpu() if isinstance(vv, torch.Tensor) else vv for kk,vv in v.items()} if v else None
        torch.save(cpu_g, self.eval_path)

    def load_eval_groups(self):
        if not os.path.exists(self.eval_path): return None
        return torch.load(self.eval_path, map_location="cpu", weights_only=True)

    def try_resume(self, model, optimizer, device):
        if not os.path.exists(self.last_path):
            print("[CKPT] No checkpoint found. Starting from epoch 1.")
            return 0, 0.0, []
        ckpt = torch.load(self.last_path, map_location=device, weights_only=True)
        model.load_state_dict(ckpt["model_state_dict"])
        try: optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        except Exception as e: print(f"[WARN] optimizer state: {e}")
        start_epoch = ckpt.get("epoch", 0)
        history = []
        if os.path.exists(self.hist_path):
            try:
                with open(self.hist_path, "r") as f: history = json.load(f)
            except Exception: pass
        scores = [h.get("Score", 0.4*h.get("Recall@K",0)) for h in history if "Score" in h or "Recall@K" in h]
        best_score = max(scores) if scores else 0.0
        print(f"[CKPT] Resume from epoch {start_epoch + 1}, best_score={best_score:.4f}")
        return start_epoch, best_score, history

ckpt_mgr = ColabCheckpointManager(weight_dir=CFG["SAVE_DIR"], data_dir=CFG["DATA_DIR"])
print("[OK] CheckpointManager ready.")

[OK] CheckpointManager ready.


In [11]:
def build_full_eval_groups(val_edges_t):
    """
    Full validation set — keeps original distribution.
    Used for early stopping and best checkpoint selection.
    """
    val_u = val_edges_t[0].cpu()
    val_i = val_edges_t[1].cpu()
    print(f"  [FULL VAL] {len(val_u):,} samples")
    return {"FULL": {"u": val_u, "i": val_i}}

print("[OK] Eval group builder ready.")

[OK] Eval group builder ready.


In [19]:
for var in ["model", "optimizer", "scaler"]:
    if var in globals(): del globals()[var]
gc.collect(); torch.cuda.empty_cache()

model = LightGCN(
    num_users=num_users, num_items=num_items,
    embed_dim=CFG["EMBED_DIM"], gcn_layers=CFG["GCN_LAYERS"],
).to(device)

scaler = torch.amp.GradScaler("cuda", enabled=(device=="cuda"))
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["LR_JOINT"],
                               weight_decay=CFG["WEIGHT_DECAY"])
resume_epoch, best_score, history = ckpt_mgr.try_resume(model, optimizer, device)

print(f"[OK] LightGCN params: {sum(p.numel() for p in model.parameters()):,}")
gc.collect(); torch.cuda.empty_cache()

[CKPT] No checkpoint found. Starting from epoch 1.
[OK] LightGCN params: 442,582,272


In [14]:
import os

last_path = os.path.join(CFG["SAVE_DIR"], "last.pth")

if os.path.exists(last_path):
    os.remove(last_path)
    print(f"[INFO] Removed corrupted checkpoint: {last_path}")
else:
    print(f"[INFO] Checkpoint not found at: {last_path}")

[INFO] Checkpoint not found at: /content/drive/MyDrive/tarecmind/weights_lightgcn_warm_only/last.pth


In [15]:
print("[INFO] Building full validation groups for checkpoint selection...")
val_full_groups = build_full_eval_groups(val_edges_t)

print("[INFO] Building train mask dict...")
df_edges = pd.DataFrame({
    "u": train_edge_index[0].cpu().numpy(),
    "i": train_edge_index[1].cpu().numpy()
})
train_mask_dict = df_edges.groupby("u")["i"].apply(
    lambda x: torch.tensor(x.values, dtype=torch.long, device="cpu")
).to_dict()
del df_edges
print(f"[OK] Train mask for {len(train_mask_dict):,} users.")
gc.collect(); torch.cuda.empty_cache()

[INFO] Building full validation groups for checkpoint selection...
  [FULL VAL] 1,774,977 samples
[INFO] Building train mask dict...
[OK] Train mask for 1,847,662 users.


In [20]:
CFG["WARMUP_EPOCHS"] = 0
CFG["PATIENCE"]      = 5

# Do NOT override resume_epoch here — it is set by ckpt_mgr.try_resume().
# Overriding would reset training to epoch 1 even when a valid checkpoint exists.
if resume_epoch == 0:
    best_score = 0.0
    history    = []
patience_cnt = 0

if resume_epoch > 0:
    for g in optimizer.param_groups:
        g.setdefault('initial_lr', g['lr'])

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG["EPOCHS"], eta_min=1e-5, last_epoch=resume_epoch-1
)

steps_per_epoch = (n_train_edges + CFG["BATCH_SIZE"] - 1) // CFG["BATCH_SIZE"]
print(f"\n[INIT] Training LightGCN (Epoch {resume_epoch+1}/{CFG['EPOCHS']})")
print("[INFO] Each epoch uses torch.randperm — every train edge visited once per epoch.")

for epoch in range(resume_epoch, CFG["EPOCHS"]):
    t0 = time.time()
    model.train()
    total_loss = 0.0
    n_processed = 0

    perm = torch.randperm(n_train_edges, device=device)

    with tqdm(total=n_train_edges,
              desc=f"Ep {epoch+1:02d} [LightGCN BPR]", ncols=100) as pbar:
        optimizer.zero_grad(set_to_none=True)
        pending_accum_steps = 0

        for step, start in enumerate(range(0, n_train_edges, CFG["BATCH_SIZE"])):
            end = min(start + CFG["BATCH_SIZE"], n_train_edges)
            idx = perm[start:end]

            u = train_edge_index[0, idx]
            p = train_edge_index[1, idx]
            n = sample_negatives_uniform(len(u), warm_item_ids, device)

            with torch.amp.autocast("cuda", enabled=(device=="cuda")):
                f_u, f_p, f_n = model(sparse_adj, u, p, n)
                raw_loss = bpr_loss(f_u, f_p, f_n)
                loss = raw_loss / CFG["ACCUM_STEPS"]

            scaler.scale(loss).backward()
            pending_accum_steps += 1

            is_accum_boundary = (pending_accum_steps >= CFG["ACCUM_STEPS"])
            is_last_step      = (step == steps_per_epoch - 1)

            if is_accum_boundary or is_last_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                pending_accum_steps = 0

            total_loss  += raw_loss.item() * len(u)
            n_processed += len(u)
            pbar.update(len(u))

    scheduler.step()
    avg_loss = total_loss / max(n_processed, 1)
    print(f"[TRAIN] Ep {epoch+1:02d} | Loss={avg_loss:.4f} | SeenEdges={n_processed:,}/{n_train_edges:,}")

    metrics  = {}
    is_best  = False

    if (epoch+1) % CFG["EVAL_EVERY"] == 0:
        print("\n[VAL-FULL] User-level early stopping and best checkpoint selection.")
        res_full = evaluate_lightgcn(
            model, sparse_adj, item_pop_group_t, val_full_groups,
            num_users, num_items, Ks=(20, 40), device=device,
            train_mask_dict=train_mask_dict,
            warm_item_mask=warm_item_mask
        )
        print_eval_results(epoch+1, res_full, Ks=(20, 40))

        # Checkpoint chính vẫn chọn theo NDCG@20 trên full validation,
        # nhưng bây giờ là user-level NDCG@20 thay vì interaction-level.
        overall_ndcg20 = res_full["OVERALL"]["NDCG@20"]
        score          = overall_ndcg20

        metrics = {
            "EvalLevel":             "user-level",
            "Recall@20":             res_full["OVERALL"]["Recall@20"],
            "NDCG@20":               res_full["OVERALL"]["NDCG@20"],
            "Recall@40":             res_full["OVERALL"]["Recall@40"],
            "NDCG@40":               res_full["OVERALL"]["NDCG@40"],
            "TailRecall@20":         res_full.get("TAIL", {}).get("Recall@20", 0),
            "TailNDCG@20":           res_full.get("TAIL", {}).get("NDCG@20", 0),
            "TailRecall@40":         res_full.get("TAIL", {}).get("Recall@40", 0),
            "TailNDCG@40":           res_full.get("TAIL", {}).get("NDCG@40", 0),
            "Score":                 score,
            "Coverage@20":           res_full.get("CoverageByK", {}).get(20, 0),
            "TailCoverage@20":       res_full.get("TailCoverageByK", {}).get(20, 0),
            "AvgPopularity@20":      res_full.get("AvgPopularityByK", {}).get(20, 0),
            "Coverage@40":           res_full.get("CoverageByK", {}).get(40, 0),
            "TailCoverage@40":       res_full.get("TailCoverageByK", {}).get(40, 0),
            "AvgPopularity@40":      res_full.get("AvgPopularityByK", {}).get(40, 0),
            # Alias cũ để không vỡ các phần đọc checkpoint/report cũ.
            "Recall@K":              res_full["OVERALL"]["Recall@20"],
            "NDCG@K":                res_full["OVERALL"]["NDCG@20"],
            "TailRecall@K":          res_full.get("TAIL", {}).get("Recall@20", 0),
            "TailNDCG@K":            res_full.get("TAIL", {}).get("NDCG@20", 0),
            "Coverage@K":            res_full.get("CoverageByK", {}).get(20, 0),
            "TailCoverage@K":        res_full.get("TailCoverageByK", {}).get(20, 0),
            "AvgPopularity@K":       res_full.get("AvgPopularityByK", {}).get(20, 0),
        }

        if score > best_score:
            best_score   = score
            is_best      = True
            patience_cnt = 0
            print(f"  NEW BEST: User-level NDCG@20={score:.4f}")
        else:
            patience_cnt += 1
            print(f"  Patience {patience_cnt}/{CFG['PATIENCE']}")

    history.append({"epoch": epoch+1, "loss": avg_loss, **metrics})
    ckpt_mgr.save(model, optimizer, epoch+1, metrics, history, CFG, is_best=is_best)

    if patience_cnt >= CFG["PATIENCE"]:
        print(f"\n[STOP] Early stopping at epoch {epoch+1}.")
        break

print(f"\nTRAINING COMPLETE! Best User-level NDCG@20={best_score:.4f}")



[INIT] Training LightGCN (Epoch 1/50)
[INFO] Each epoch uses torch.randperm — every train edge visited once per epoch.


Ep 01 [LightGCN BPR]:   0%|                                            | 0/13964281 [00:00<?, ?it/s]

[TRAIN] Ep 01 | Loss=0.4464 | SeenEdges=13,964,281/13,964,281


Ep 02 [LightGCN BPR]:   0%|                                            | 0/13964281 [00:00<?, ?it/s]

[TRAIN] Ep 02 | Loss=0.2475 | SeenEdges=13,964,281/13,964,281

[VAL-FULL] User-level early stopping and best checkpoint selection.
[USER-LEVEL VAL] Users=1,774,977 | Positives=1,774,977


...

Ep 44 [LightGCN BPR]:   0%|                                            | 0/13964281 [00:00<?, ?it/s]

[TRAIN] Ep 44 | Loss=0.0634 | SeenEdges=13,964,281/13,964,281

[VAL-FULL] User-level early stopping and best checkpoint selection.
[USER-LEVEL VAL] Users=1,774,977 | Positives=1,774,977


User-level VAL:   0%|                                                     | 0/27735 [00:00<?, ?it/s]


[VAL WARM-ONLY USER-LEVEL] Epoch 44
  TAIL       | users= 154,009 | pos= 154,009 | R@20=0.0001 | N@20=0.0000 | R@40=0.0001 | N@40=0.0000
  OVERALL    | users=1,774,977 | pos=1,774,977 | R@20=0.0277 | N@20=0.0116 | R@40=0.0416 | N@40=0.0144
  Coverage@20=0.0074 | TailCov@20=0.0068 | AvgPop@20=185.9
  Coverage@40=0.0135 | TailCov@40=0.0121 | AvgPop@40=143.3
  Warm candidates=1,042,121
  [TAIL] R@20=0.0001 | N@20=0.0000 | R@40=0.0001 | N@40=0.0000 | TailCov@20=0.0068 | TailCov@40=0.0121
  NEW BEST: User-level NDCG@20=0.0116


RuntimeError: [enforce fail at inline_container.cc:743] . open file failed with strerror: Operation not supported

In [21]:
gc.collect(); torch.cuda.empty_cache()

In [22]:
print("\n--- LOAD BEST MODEL ---")
if os.path.exists(ckpt_mgr.best_path):
    ckpt = torch.load(ckpt_mgr.best_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"[OK] Loaded best model epoch {ckpt['epoch']}")
else:
    print("[WARN] No best model found, using current weights.")

print("\n--- LOADING TEST SET ---")
df_test = load_dataset(
    "chuongdo1104/amazon-2023-bronze",
    data_dir="bronze/bronze_test.parquet",
    split="train"
).to_pandas()
u_col = next(c for c in ["user_idx","user_id","reviewer_id"] if c in df_test.columns)
i_col = next(c for c in ["item_idx","item_id","parent_asin"] if c in df_test.columns)
if df_test[u_col].dtype == object:
    print("[INFO] Mapping string IDs to integer IDs...")
    df_user_map = pd.read_parquet(download_hf("gold/gold_user_id_map.parquet"))
    df_item_map = pd.read_parquet(download_hf("gold/gold_item_id_map.parquet"))
    u_map = dict(zip(df_user_map["reviewer_id"], df_user_map["user_idx"]))
    i_map = dict(zip(df_item_map["parent_asin"], df_item_map["item_idx"]))
    df_test["user_idx"] = df_test[u_col].map(u_map)
    df_test["item_idx"] = df_test[i_col].map(i_map)
    df_test = df_test.dropna(subset=["user_idx","item_idx"])

test_users_t = torch.tensor(df_test["user_idx"].values.astype(int), dtype=torch.long)
test_items_t = torch.tensor(df_test["item_idx"].values.astype(int), dtype=torch.long)

# Warm-only test positives:
# 1) user/item phải hợp lệ
# 2) positive item phải có train_freq > 0
base_mask = (test_users_t >= 0) & (test_users_t < num_users) & (test_items_t >= 0) & (test_items_t < num_items)
warm_test_mask = torch.zeros_like(base_mask, dtype=torch.bool)
valid_items = test_items_t[base_mask]
warm_test_mask[base_mask] = warm_item_mask_cpu[valid_items]
mask = base_mask & warm_test_mask

TEST_SIZE_RAW = int(len(test_users_t))
test_users_filt = test_users_t[mask].to(device)
test_items_filt = test_items_t[mask].to(device)
TEST_SIZE_WARM = int(len(test_users_filt))
print(f"[FILTER] Test warm-only: {TEST_SIZE_RAW:,} -> {TEST_SIZE_WARM:,}")
print(f"[FILTER] Removed cold/invalid test positives: {TEST_SIZE_RAW - TEST_SIZE_WARM:,}")
assert (item_train_freq_cpu[test_items_filt.cpu()] > 0).all(), "Test still contains cold positives!"
print(f"[OK] Test set warm-only: {len(test_users_filt):,} interactions.")

print("\n--- TEST MASKING ---")

# train_user_dict: dùng khi đánh giá validation
# test_mask_dict: dùng khi đánh giá test, gồm train + validation items
# Lưu ý: tuyệt đối không đưa test item vào mask_dict, vì test item là đáp án cần tìm.

train_user_dict = {
    int(u): items.detach().cpu().long()
    for u, items in train_mask_dict.items()
}

df_val_mask = pd.DataFrame({
    "u": val_edges_t[0].detach().cpu().numpy(),
    "i": val_edges_t[1].detach().cpu().numpy(),
})

val_user_dict = df_val_mask.groupby("u")["i"].apply(
    lambda x: torch.tensor(x.values, dtype=torch.long, device="cpu")
).to_dict()
val_user_dict = {
    int(u): items.detach().cpu().long()
    for u, items in val_user_dict.items()
}
del df_val_mask

test_mask_dict = {
    u: items.clone()
    for u, items in train_user_dict.items()
}

for u, val_items in val_user_dict.items():
    if u in test_mask_dict:
        test_mask_dict[u] = torch.unique(torch.cat([test_mask_dict[u], val_items]))
    else:
        test_mask_dict[u] = val_items.clone()

print(f"[OK] Train mask dict      : {len(train_user_dict):,} users.")
print(f"[OK] Validation mask dict : {len(val_user_dict):,} users.")
print(f"[OK] Test mask dict       : {len(test_mask_dict):,} users. Test mask = train + validation.")

item_groups_eval = item_pop_group_t.to(device)

# Kiểm tra cuối: validation/test không còn cold positive.
val_cold_count  = int((item_train_freq_cpu[val_edges_t[1].cpu()] == 0).sum().item())
test_cold_count = int((item_train_freq_cpu[test_items_filt.cpu()] == 0).sum().item())
print(f"[CHECK] Cold positives in validation: {val_cold_count}")
print(f"[CHECK] Cold positives in test      : {test_cold_count}")
print(f"[CHECK] Warm candidate items        : {num_warm_items:,}")
print(f"[CHECK] Cold candidate items removed: {num_cold_items:,}")
assert val_cold_count == 0
assert test_cold_count == 0

gc.collect(); torch.cuda.empty_cache()


--- LOAD BEST MODEL ---
[OK] Loaded best model epoch 42

--- LOADING TEST SET ---


bronze/bronze_test.parquet/part-00000-2a(…):   0%|          | 0.00/32.1M [00:00<?, ?B/s]

bronze/bronze_test.parquet/part-00001-2a(…):   0%|          | 0.00/31.9M [00:00<?, ?B/s]

bronze/bronze_test.parquet/part-00002-2a(…):   0%|          | 0.00/64.0M [00:00<?, ?B/s]

bronze/bronze_test.parquet/part-00003-2a(…):   0%|          | 0.00/63.9M [00:00<?, ?B/s]

...

[INFO] Mapping string IDs to integer IDs...
[INFO] Downloading gold/gold_user_id_map.parquet...


gold/gold_user_id_map.parquet:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

[INFO] Downloading gold/gold_item_id_map.parquet...


gold/gold_item_id_map.parquet:   0%|          | 0.00/97.0M [00:00<?, ?B/s]

[FILTER] Test warm-only: 1,847,662 -> 1,757,753
[FILTER] Removed cold/invalid test positives: 89,909
[OK] Test set warm-only: 1,757,753 interactions.

--- TEST MASKING ---
[OK] Train mask dict      : 1,847,662 users.
[OK] Validation mask dict : 1,774,977 users.
[OK] Test mask dict       : 1,847,662 users. Test mask = train + validation.
[CHECK] Cold positives in validation: 0
[CHECK] Cold positives in test      : 0
[CHECK] Warm candidate items        : 1,042,121
[CHECK] Cold candidate items removed: 567,891


In [23]:
print("\n--- BUILDING FINAL REPRESENTATIONS ---")
model.eval()
gc.collect(); torch.cuda.empty_cache()

with torch.no_grad():
    with torch.amp.autocast("cuda", enabled=(device=="cuda")):
        z_G_all = model.forward_gcn(sparse_adj)
    user_G_all, item_G_all = z_G_all.split([num_users, num_items])
    item_final_all = item_G_all
    user_final_all = user_G_all.cpu()
    del z_G_all, user_G_all, item_G_all

gc.collect(); torch.cuda.empty_cache()
print("[OK] Representations ready.")

ITEM_NAMES = {2: "Item_TAIL"}

test_u_idx       = test_users_filt.cpu().numpy().astype(np.int64)
test_i_idx       = test_items_filt.cpu().numpy().astype(np.int64)
test_user_dict   = group_by_user_items(test_u_idx, test_i_idx, deduplicate=True)
test_user_dict   = _filter_eval_positives_seen_in_mask(test_user_dict, test_mask_dict)

print(f"[USER-LEVEL TEST] Users={len(test_user_dict):,} | Positives={sum(len(v) for v in test_user_dict.values()):,}")

def run_final_evaluation(Ks=(20, 40), user_batch=128):
    Ks = sorted(list(Ks))
    max_k_requested = max(Ks)
    topk_eval = min(max_k_requested, num_warm_items)

    eval_user_ids = np.array(list(test_user_dict.keys()), dtype=np.int64)
    n_eval_users = int(len(eval_user_ids))

    group_names = ["OVERALL", "Item_TAIL"]
    accum = _empty_user_level_accum(Ks, group_names)

    recommended_mask = {
        K: torch.zeros(num_items, dtype=torch.bool, device=device)
        for K in Ks
    }

    tail_mask_eval = (item_groups_eval == 2) & warm_item_mask
    pop_g_cpu = item_groups_eval.detach().cpu().long()

    discounts = 1.0 / np.log2(np.arange(2, topk_eval + 2, dtype=np.float64))

    print(f"\n--- FULL-RANKING TEST WARM-ONLY USER-LEVEL (Ks={Ks}) ---")
    print(f"Candidate set: {num_warm_items:,} warm items | cold removed: {num_cold_items:,}")
    print(f"Eval users: {n_eval_users:,} | eval positives: {sum(len(v) for v in test_user_dict.values()):,}")

    with torch.no_grad():
        for start in tqdm(range(0, n_eval_users, user_batch), desc="User-level TEST", ncols=100):
            end = min(start + user_batch, n_eval_users)
            bu_np = eval_user_ids[start:end]
            bu_cpu = torch.tensor(bu_np, dtype=torch.long)

            u_embs = user_final_all[bu_cpu].to(device)

            with torch.amp.autocast("cuda", enabled=(device=="cuda")):
                scores = torch.mm(u_embs, item_final_all.t())
                mask_value = torch.finfo(scores.dtype).min

                # Candidate set warm-only: loại toàn bộ cold item khỏi ranking.
                scores[:, cold_item_mask] = mask_value

            # Mask train + validation items.
            # Không mask test positives, vì đó là đáp án cần tìm.
            for row, u_val in enumerate(bu_np):
                seen = test_mask_dict.get(int(u_val))
                if seen is not None and len(seen) > 0:
                    scores[row, seen.to(device)] = mask_value

            _, topk_idx = torch.topk(scores, topk_eval, dim=1)
            topk_np = topk_idx.detach().cpu().numpy()

            for K in Ks:
                use_k = min(K, topk_eval)
                recommended_mask[K][topk_idx[:, :use_k].reshape(-1)] = True

            for row, u_val in enumerate(bu_np):
                pos_items = test_user_dict[int(u_val)].astype(np.int64)
                top_items = topk_np[row]

                for K in Ks:
                    _add_user_level_metric(accum, "OVERALL", K, top_items, pos_items, discounts)

                pos_groups = pop_g_cpu[torch.tensor(pos_items, dtype=torch.long)].numpy()
                tail_pos = pos_items[pos_groups == 2]
                if len(tail_pos) > 0:
                    for K in Ks:
                        _add_user_level_metric(accum, "Item_TAIL", K, top_items, tail_pos, discounts)

            del u_embs, scores

    cov, tcov, apop = {}, {}, {}
    n_tail = int(tail_mask_eval.sum().item())

    for K in Ks:
        rm = recommended_mask[K] & warm_item_mask
        nr = int(rm.sum().item())
        cov[K] = nr / num_warm_items if num_warm_items > 0 else 0
        tcov[K] = (rm & tail_mask_eval).sum().item() / n_tail if n_tail > 0 else 0
        apop[K] = item_train_freq_t[rm].float().mean().item() if nr > 0 else 0

    results = {}
    for g in group_names:
        results[g] = {}
        for K in Ks:
            n_users = accum[g][K]["n_users"]
            results[g][f"Recall@{K}"] = accum[g][K]["recall_sum"] / max(n_users, 1)
            results[g][f"NDCG@{K}"] = accum[g][K]["ndcg_sum"] / max(n_users, 1)
            results[g][f"Hits@{K}"] = accum[g][K]["hits"]
            results[g][f"n_users@{K}"] = n_users
            results[g][f"n_pos@{K}"] = accum[g][K]["n_pos"]

    results["CoverageByK"] = cov
    results["TailCoverageByK"] = tcov
    results["AvgPopularityByK"] = apop
    results["EvalLevel"] = "user-level"

    print("\n" + "="*100)
    print("FINAL TEST — WARM-ONLY FULL-RANKING USER-LEVEL")
    print("="*100)

    for g in ["OVERALL", "Item_TAIL"]:
        if g not in results:
            continue

        n_users = results[g].get(f"n_users@{Ks[0]}", 0)
        n_pos = results[g].get(f"n_pos@{Ks[0]}", 0)
        parts = [f"{g:<16} | users={n_users:>8,} | pos={n_pos:>8,}"]

        for K in Ks:
            r = results[g].get(f"Recall@{K}", 0)
            nd = results[g].get(f"NDCG@{K}", 0)
            parts.append(f"R@{K}={r:.4f} N@{K}={nd:.4f}")

        print(" | ".join(parts))

    for K in Ks:
        print(f"Coverage@{K}={cov[K]:.4f} | TailCov@{K}={tcov[K]:.4f} | AvgPop@{K}={apop[K]:.1f}")

    print(f"Warm candidates={num_warm_items:,} | Cold candidates removed={num_cold_items:,}")

    tail_r20 = results.get("Item_TAIL", {}).get("Recall@20", 0)
    tail_n20 = results.get("Item_TAIL", {}).get("NDCG@20", 0)
    tail_r40 = results.get("Item_TAIL", {}).get("Recall@40", 0)
    tail_n40 = results.get("Item_TAIL", {}).get("NDCG@40", 0)
    print(
        f"\n  [TAIL] TailRecall@20={tail_r20:.4f} | TailNDCG@20={tail_n20:.4f} "
        f"| TailRecall@40={tail_r40:.4f} | TailNDCG@40={tail_n40:.4f} "
        f"| TailCoverage@20={tcov.get(20, 0):.4f} | TailCoverage@40={tcov.get(40, 0):.4f}"
    )
    print("="*100)

    return results

results_full = run_final_evaluation(Ks=(20, 40))



--- BUILDING FINAL REPRESENTATIONS ---
[OK] Representations ready.
[USER-LEVEL TEST] Users=1,757,753 | Positives=1,757,753

--- FULL-RANKING TEST WARM-ONLY USER-LEVEL (Ks=[20, 40]) ---
Candidate set: 1,042,121 warm items | cold removed: 567,891
Eval users: 1,757,753 | eval positives: 1,757,753


User-level TEST:   0%|                                                    | 0/13733 [00:00<?, ?it/s]


FINAL TEST — WARM-ONLY FULL-RANKING USER-LEVEL
OVERALL          | users=1,757,753 | pos=1,757,753 | R@20=0.0199 N@20=0.0081 | R@40=0.0310 N@40=0.0103
Item_TAIL        | users= 173,508 | pos= 173,508 | R@20=0.0000 N@20=0.0000 | R@40=0.0000 N@40=0.0000
Coverage@20=0.0073 | TailCov@20=0.0067 | AvgPop@20=187.0
Coverage@40=0.0133 | TailCov@40=0.0120 | AvgPop@40=143.5
Warm candidates=1,042,121 | Cold candidates removed=567,891

  [TAIL] TailRecall@20=0.0000 | TailNDCG@20=0.0000 | TailRecall@40=0.0000 | TailNDCG@40=0.0000 | TailCoverage@20=0.0067 | TailCoverage@40=0.0120


In [ ]:
# Sampled evaluation (1+100) removed.

In [ ]:
# Sampled evaluation (1+1000) removed.

In [24]:
import json, time
import numpy as np

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

print("="*70)
print(f"{'FINAL REPORT — ' + VARIANT_NAME:^70}")
print("="*70)

final_metrics = {
    "variant":   VARIANT_NAME,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "protocol":  "warm_only_no_cold_start_user_level_full_ranking",
    "eval_level": "user-level",
    "ks": [20, 40],
    "num_items_total": int(num_items),
    "num_warm_items": int(num_warm_items),
    "num_cold_items_removed": int(num_cold_items),
    "val_size_warm_interactions": int(VAL_SIZE),
    "test_size_warm_interactions": int(len(test_users_filt)),
    "test_size_warm_users": int(len(test_user_dict)) if "test_user_dict" in globals() else None,
}

if "results_full" in globals() and isinstance(results_full, dict):
    for group_name in ["OVERALL", "Item_TAIL"]:
        if group_name in results_full:
            for K in [20, 40]:
                final_metrics[f"{group_name}_Recall@{K}"] = results_full[group_name].get(f"Recall@{K}", 0)
                final_metrics[f"{group_name}_NDCG@{K}"] = results_full[group_name].get(f"NDCG@{K}", 0)
                final_metrics[f"{group_name}_n_users@{K}"] = results_full[group_name].get(f"n_users@{K}", 0)
                final_metrics[f"{group_name}_n_pos@{K}"] = results_full[group_name].get(f"n_pos@{K}", 0)

    for K in [20, 40]:
        final_metrics[f"Coverage@{K}"] = results_full.get("CoverageByK", {}).get(K, 0)
        final_metrics[f"TailCoverage@{K}"] = results_full.get("TailCoverageByK", {}).get(K, 0)
        final_metrics[f"AvgPopularity@{K}"] = results_full.get("AvgPopularityByK", {}).get(K, 0)

with open(os.path.join(CFG["DATA_DIR"], f"report_{VARIANT_NAME}.json"), "w") as f:
    json.dump(final_metrics, f, indent=2, cls=NpEncoder)
print(f"[OK] Report saved to {CFG['DATA_DIR']}/report_{VARIANT_NAME}.json")


                  FINAL REPORT — lightgcn_warm_only                   
[OK] Report saved to /content/drive/MyDrive/tarecmind/data_lightgcn_warm_only/report_lightgcn_warm_only.json


In [25]:
import time
print("Pipeline complete. Disconnecting in 10s...")
time.sleep(10)
try:
    from google.colab import runtime
    runtime.unassign()
except:
    print("[INFO] Not in Colab.")

Pipeline complete. Disconnecting in 10s...
